# Keras to PyTorch: Antarctic Field Guide

## Read a familiar Keras workflow in PyTorch, one operation at a time

You already know the underlying ideas from prerequisites 00–02: matrices carry data, train/validation/test splits protect evaluation, dense layers own weights, a loss measures mistakes, and backpropagation assigns gradients. This notebook changes the **framework vocabulary**, not the theory.

Our project is **Antarctic Field Guide**: predict a Palmer penguin's species from field measurements. The task is deliberately tabular, so PyTorch tensor, module, loss, optimizer, and inference contracts become visible before CNN or sequence-specific APIs add extra dimensions.

| Part | Keras idea you know | PyTorch translation | Project evidence |
|---|---|---|---|
| 0 | Set seeds, choose data | Explicit seeds and `device` | Reproducible data manifest |
| 1 | Tensor conversion | `torch.as_tensor`, explicit `float32` / `long` | Feature and label contracts |
| 2 | `Dense` / `Sequential` | `nn.Linear` / `nn.Sequential` | Fixed-weight parity check |
| 3 | `from_logits=True` | `nn.CrossEntropyLoss` consumes raw logits | Logits-contract experiment |
| 4 | `GradientTape` / `fit()` | Explicit `zero_grad -> backward -> step` | Gradient and loss curves |
| 5 | `predict()` | `eval()` + `torch.no_grad()` | Held-out species audit |
| 6 | Change one variable | Controlled ablations | Measured project decisions |

**Source and rights.** The [Palmer Penguins dataset](https://allisonhorst.github.io/palmerpenguins/) is published under CC0. The `palmerpenguins` Python package bundles the numeric measurements used here, so the project has no data download, no score-edition ambiguity, and no hidden label leakage.

![Seven-stage Keras-to-PyTorch workflow from Palmer Penguins data through typed tensors, model construction, logits and loss, explicit training, evaluation, and controlled ablations.](images/4.png)

*The project keeps the same learning goals while making PyTorch's tensor, training, and evaluation contracts explicit.*

---

## What You Bring, What You Learn

| Already established in `genai-prerequisites` | New PyTorch practice here | Intentionally not taught here |
|---|---|---|
| Dense layers, activation functions, losses, optimizers | Tensor dtypes, `nn.Module`, `nn.Linear` weight layout, `.backward()`, train/eval modes | RNN hidden state, BPTT, LSTM gates, attention, audio generation |
| Keras `Sequential`, `GradientTape`, `fit`, `predict` | Explicit training loop, gradients accumulating until cleared, `state_dict` vocabulary | `DataLoader`, checkpointing workflow, dropout, mini-batching |

Every Keras snippet is a **reference cell**. The executable cells are PyTorch-only, matching the rest of this GenAI track.


In [ ]:
#  Setup — install only packages absent from the active kernel
import importlib.util
import subprocess
import sys

# Only install packages that aren't already importable in this kernel
for package, module in [("palmerpenguins", "palmerpenguins"), ("seaborn", "seaborn")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

print("Setup check complete. The next cell imports the PyTorch-only runtime.")

In [ ]:
#  Imports, deterministic seeds, and device selection
import random

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
from palmerpenguins import load_penguins

SEED = 17

# Seed Python, NumPy, and PyTorch separately since each owns its own RNG
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer a GPU if visible to PyTorch, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device:  {device}")
print("-> Keras users: explicit three-library seeding replaces keras.utils.set_random_seed(SEED).")

### Keras reference — reproducibility and model device

```python
keras.utils.set_random_seed(SEED)
# TensorFlow usually selects an available GPU automatically.
```

PyTorch keeps both choices explicit: seed Python, NumPy, and PyTorch, then move the model and tensors to one `device`. That makes placement visible in every training cell.


---

## Part 1 — Build a CC0 Project Dataset

Antarctic Field Guide predicts species from four measured physical features: bill length, bill depth, flipper length, and body mass. The label is a species ID. We remove rows with missing measurements before splitting, then fit preprocessing statistics on training data only.

### Predict first

Will the labels be perfectly balanced?

1. Exactly balanced: each species has identical representation.
2. Skewed: the field dataset's actual collection determines class balance.
3. There will be no usable rows because the data are not numeric.


In [ ]:
#  Data extraction — CC0 Palmer Penguins measurements packaged with palmerpenguins
FEATURE_COLUMNS = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

# Load the bundled CC0 Palmer Penguins measurements
penguins = load_penguins()

# Drop rows missing any feature or the label before splitting
complete_penguins = penguins.dropna(subset=FEATURE_COLUMNS + ["species"]).reset_index(drop=True)

# Collect the sorted, distinct species names to fix a stable class order
species_names = sorted(complete_penguins["species"].unique())

# Map each species name to a stable integer class ID
species_to_id = {species: index for index, species in enumerate(species_names)}

# Convert features to float32 and labels to int64, the NumPy dtypes the tensor step expects
X_raw = complete_penguins[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
y_raw = complete_penguins["species"].map(species_to_id).to_numpy(dtype=np.int64)

# Fail fast if the CC0 package ever ships a materially smaller dataset
if len(complete_penguins) < 300:
    raise RuntimeError(f"Expected at least 300 complete penguin records; found {len(complete_penguins)}.")

print("Source: Palmer Penguins (CC0), bundled by palmerpenguins")
print(f"Rows retained after missing-value removal: {len(complete_penguins)}")
print(f"Feature shape: {X_raw.shape}; labels shape: {y_raw.shape}")
print("Class balance:", complete_penguins["species"].value_counts().to_dict())
print("-> The target is species; all four input columns are numeric field measurements.")


In [ ]:
#  Data health check — inspect the CC0 field-data class balance and feature scales
# Plot species class counts alongside two feature-value histograms to sanity-check the data
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=complete_penguins, x="species", ax=axes[0], color="#4C78A8")
axes[0].set(title="Species labels in the CC0 dataset", xlabel="Species", ylabel="Penguins")
axes[0].tick_params(axis="x", rotation=20)
axes[1].hist(X_raw[:, 0], bins=20, alpha=0.8, label="bill length (mm)")
axes[1].hist(X_raw[:, 3], bins=20, alpha=0.6, label="body mass (g)")
axes[1].set(title="Two physical measurement distributions", xlabel="Feature value", ylabel="Penguins")
axes[1].legend()
plt.tight_layout()
plt.show()

# Confirm the expected feature/label shape before moving on
assert X_raw.shape[1] == 4 and len(species_names) == 3
print("-> The project has three observed species and four numeric features per penguin.")

#### What just happened — and what's missing

We now have a real, reproducible CC0 classification dataset, with a visible missing-value policy. But NumPy arrays are not yet model inputs. The next translation is the one Keras usually performs quietly: feature floats and class IDs need distinct tensor dtypes.


---

## Part 2 — From NumPy Arrays to PyTorch Tensor Contracts

### Keras reference — implicit tensor conversion

```python
X = tf.convert_to_tensor(X_scaled, dtype=tf.float32)
y = tf.convert_to_tensor(y_labels, dtype=tf.int32)
```

Keras layers often accept NumPy arrays directly. In PyTorch, converting deliberately exposes the dtype contract: features are `float32`; class IDs are `long` (`int64`).


In [ ]:
#  Deterministic stratified split and train-only normalization
indices = np.arange(len(y_raw))
rng = np.random.default_rng(SEED)
train_indices, test_indices = [], []

# Split each species independently so the 80/20 ratio holds within every class
for label in range(len(species_names)):
    label_indices = indices[y_raw == label].copy()
    rng.shuffle(label_indices)
    cutoff = int(len(label_indices) * 0.8)
    train_indices.extend(label_indices[:cutoff])
    test_indices.extend(label_indices[cutoff:])
train_indices = np.array(sorted(train_indices))
test_indices = np.array(sorted(test_indices))
X_train_raw, X_test_raw = X_raw[train_indices], X_raw[test_indices]
y_train_np, y_test_np = y_raw[train_indices], y_raw[test_indices]

# Fit normalization statistics on the training split only, to avoid test-set leakage
mean = X_train_raw.mean(axis=0, keepdims=True)
std = X_train_raw.std(axis=0, keepdims=True)

# Guard against divide-by-zero for any near-constant feature
std[std < 1e-6] = 1.0
X_train_np, X_test_np = (X_train_raw - mean) / std, (X_test_raw - mean) / std

# Convert to the PyTorch dtype/device contract: float32 features, long class IDs
X_train = torch.as_tensor(X_train_np, dtype=torch.float32, device=device)
X_test = torch.as_tensor(X_test_np, dtype=torch.float32, device=device)
y_train = torch.as_tensor(y_train_np, dtype=torch.long, device=device)
y_test = torch.as_tensor(y_test_np, dtype=torch.long, device=device)

# Confirm the dtype and shape contract holds before training
assert X_train.dtype == torch.float32 and y_train.dtype == torch.long
assert X_train.shape[1] == 4 and X_train.shape[0] == y_train.shape[0]
print(f"X_train: {tuple(X_train.shape)}, {X_train.dtype}")
print(f"y_train: {tuple(y_train.shape)}, {y_train.dtype}")
print("-> float features + long class IDs are the PyTorch contract for nn.Linear + CrossEntropyLoss.")

### Predict first — can Keras weights be copied directly into `nn.Linear`?

Keras stores a `Dense(3)` kernel as `(in_features, out_features)`. PyTorch stores `nn.Linear(in_features, 3).weight` as `(out_features, in_features)`.

1. Yes, identical layout: copy the matrix unchanged.
2. No, transpose it: both frameworks compute the same affine function but expose different storage orientation.
3. Neither: PyTorch cannot reproduce a Keras dense layer exactly.


### Keras reference — a dense layer

```python
layer = keras.layers.Dense(3, use_bias=True)
logits = layer(features)  # kernel: (in_features, 3)
```

The activation is intentionally absent: both frameworks return **raw logits** from the last layer, leaving softmax to the loss/inference step.


In [ ]:
#  Dense-layer parity proof — same affine math, transposed weight storage
torch.manual_seed(SEED)
keras_style_kernel = torch.tensor([[0.20, -0.10], [0.35, 0.40], [-0.25, 0.15]])
keras_style_bias = torch.tensor([0.05, -0.20])
probe = torch.tensor([[1.0, 2.0, -1.0]])
linear = nn.Linear(3, 2)

# Copy the Keras-style kernel in transposed form to match nn.Linear's (out, in) storage
with torch.no_grad():
    linear.weight.copy_(keras_style_kernel.T)
    linear.bias.copy_(keras_style_bias)

# Compute the same affine map by hand, using the untransposed Keras-orientation kernel
keras_math = probe @ keras_style_kernel + keras_style_bias
pytorch_math = linear(probe)

# Confirm both frameworks produce identical outputs despite the storage-layout difference
assert torch.allclose(keras_math, pytorch_math)
print(f"Keras kernel shape:    {tuple(keras_style_kernel.shape)} = (in, out)")
print(f"PyTorch weight shape:  {tuple(linear.weight.shape)} = (out, in)")
print(f"Parity check: {torch.allclose(keras_math, pytorch_math)}")
print("-> Transpose at the storage boundary; the forward computation is the same affine map.")

![Keras and PyTorch implement the same affine map using transposed weight-storage layouts: Keras multiplies a row input by a three-by-two kernel, while PyTorch multiplies it by the transpose of a two-by-three weight matrix.](images/1.png)

*The forward computation is identical; only the exposed weight-storage orientation changes.*

#### What just happened — and what's missing

The Keras/PyTorch shift is not new math. It is a more explicit contract: PyTorch makes dtype, device, and weight orientation visible. Before training the project model, make the `GradientTape` to `.backward()` translation concrete on a derivative whose answer we already know.


### Keras reference — one derivative with `GradientTape`

```python
x = tf.Variable(3.0)
with tf.GradientTape() as tape:
    y = x ** 2
dy_dx = tape.gradient(y, x)  # 6.0
```

PyTorch records operations whenever an input has `requires_grad=True`. Calling `backward()` walks that recorded graph and places the result in the input's `.grad` field.


In [ ]:
#  Autograd warm-up — same derivative as GradientTape, explicit gradient storage
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()

# Confirm the analytic derivative (dy/dx = 2x = 6.0 at x=3) matches autograd's result
assert x.grad.item() == 6.0
print(f"y = x² at x={x.item():.1f} gives y={y.item():.1f}")
print(f"dy/dx stored in x.grad: {x.grad.item():.1f}")
print("-> requires_grad records the operations; backward() writes the derivative into .grad.")

In [ ]:
#  Gradient-descent warm-up — use autograd to minimize a tiny known parabola
position = torch.tensor(-2.0, requires_grad=True)
target = 5.0
trajectory = []

# Run gradient descent by hand: compute loss, backprop, step, then clear the gradient
for _ in range(80):
    loss = (position - target) ** 2
    loss.backward()
    with torch.no_grad():
        position -= 0.08 * position.grad
    position.grad.zero_()
    trajectory.append(position.item())

# Confirm the position converged to the known target
assert abs(position.item() - target) < 1e-4

# Plot the position trajectory against the known minimum
plt.figure(figsize=(7, 3))
plt.plot(trajectory, color="#E45756")
plt.axhline(target, linestyle="--", color="#54A24B", label="minimum")
plt.title("Autograd drives a scalar toward a known minimum")
plt.xlabel("Step")
plt.ylabel("Position")
plt.legend()
plt.tight_layout()
plt.show()
print(f"Position after 80 steps: {position.item():.4f} (target {target:.1f})")
print("-> The same gradient-descent loop will update every Antarctic Field Guide weight together.")

#### Watch one autograd graph live and retire

![Animation tracing a PyTorch forward pass, dynamic computation graph creation, backward gradient flow, parameter update, gradient clearing, and graph disposal](images/pytorch-autograd-graph-lifecycle.gif)

Each forward pass records the operations needed for that step. `backward()` traverses those edges in reverse and accumulates gradients; the update changes the leaf value, `zero_()` clears stored gradients, and the next iteration builds a fresh graph from the new state.

---

## Part 3 — Logits First, Probabilities Only When You Need Them

### Keras reference — a compact classifier

```python
model = keras.Sequential([
    keras.Input(shape=(4,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(3),
])
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
```

`from_logits=True` is the important part. It tells Keras to apply stable softmax-inside-cross-entropy math. PyTorch's `nn.CrossEntropyLoss()` assumes raw logits by default.


![Two-lane classifier contract: raw logits feed cross-entropy directly during training, while inference converts logits to probabilities before selecting a class; applying softmax before cross-entropy is marked incorrect.](images/2.png)

*Keep logits raw for `CrossEntropyLoss`; convert them to probabilities only for human-facing inference.*

In [ ]:
#  Antarctic Field Guide model — the readable PyTorch equivalent of Keras Sequential
HIDDEN_WIDTH = 16


# Two-layer MLP: PyTorch's explicit analogue of a Keras Sequential classifier
class AntarcticFieldGuide(nn.Module):
    def __init__(self, input_features=4, hidden_width=HIDDEN_WIDTH, classes=3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_features, hidden_width),
            nn.ReLU(),
            nn.Linear(hidden_width, classes),
        )

    def forward(self, features):
        return self.layers(features)  # (batch, 2) raw logits


model = AntarcticFieldGuide().to(device)
logits = model(X_train[:4])

# Confirm the model emits one logit row per input example, one column per class
assert logits.shape == (4, 3)
print(model)
print(f"Logit shape: {tuple(logits.shape)}; row 0 raw scores: {logits[0].detach().cpu().numpy()}")
print("-> nn.Module.forward() is the PyTorch analogue of Keras Layer.call().")

#### Why does `model(X_train[:4])` work if `model` isn't a function?

`model` is an *instance* of `AntarcticFieldGuide`, and `nn.Module` (its parent class) defines a
`__call__` method. Any Python object whose class implements `__call__` becomes callable with `()`,
so `model(...)` really means `nn.Module.__call__(model, ...)` -- not an attempt to instantiate a
class. That `__call__` doesn't just immediately run `forward()`, either: it also runs any hooks
registered on the module (used for things like activation capturing or quantization) before and
after invoking `forward()`. That's the reason for PyTorch's universal convention -- call the module
itself, never `.forward()` directly -- since bypassing `__call__` silently skips that hook machinery.



In [ ]:
#  Logits contract experiment — raw scores, not softmax probabilities, enter CrossEntropyLoss
fixed_logits = torch.tensor([[3.0, -1.0], [-0.5, 1.0]], requires_grad=True)
fixed_targets = torch.tensor([0, 1], dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()

# Correct usage: feed raw logits directly into CrossEntropyLoss
correct_loss = loss_fn(fixed_logits, fixed_targets)

# Incorrect usage: softmax before CrossEntropyLoss double-applies the normalization
incorrect_loss = loss_fn(torch.softmax(fixed_logits, dim=1), fixed_targets)

# Compare gradient magnitudes to show the "incorrect" call still runs but distorts gradients
correct_grad = torch.autograd.grad(correct_loss, fixed_logits, retain_graph=True)[0].norm().item()
incorrect_grad = torch.autograd.grad(incorrect_loss, fixed_logits)[0].norm().item()
print(f"CrossEntropyLoss(raw logits):      {correct_loss.item():.5f}; gradient norm {correct_grad:.5f}")
print(f"CrossEntropyLoss(softmax(logits)): {incorrect_loss.item():.5f}; gradient norm {incorrect_grad:.5f}")
print("-> Both calls run, but only raw logits honor the intended CrossEntropyLoss contract.")

#### What just happened — and what's missing

The final layer emits two raw scores, not probabilities. `CrossEntropyLoss` owns the stable conversion during training; `softmax` is reserved for human-facing inference. We have a model and a loss, but Keras's `fit()` has not yet revealed its work. PyTorch will make that work explicit.


---

## Part 4 — What `fit()` Hides

### Keras reference — high-level training

```python
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.02), loss=loss_fn)
history = model.fit(X_train, y_train, epochs=160, verbose=0)
```

The PyTorch loop below exposes the same four events. The order matters because PyTorch accumulates gradients in every parameter's `.grad` field until you clear them.


![Four-stage PyTorch training cycle: clear gradients, run the forward pass and compute loss, backpropagate into parameter gradients, then update parameters; a warning branch shows gradient accumulation when clearing is skipped.](images/3.png)

*PyTorch exposes the optimization cycle directly, including its intentional gradient-accumulation behavior.*

In [ ]:
#  One explicit PyTorch optimization step — the four operations behind fit()
torch.manual_seed(SEED)
step_model = AntarcticFieldGuide().to(device)
step_optimizer = torch.optim.Adam(step_model.parameters(), lr=0.02)
step_optimizer.zero_grad()
step_logits = step_model(X_train)
step_loss = loss_fn(step_logits, y_train)
step_loss.backward()

# Aggregate the L2 norm across every parameter's gradient into one scalar
gradient_norm = torch.sqrt(sum(parameter.grad.pow(2).sum() for parameter in step_model.parameters())).item()
step_optimizer.step()
print(f"Loss before update: {step_loss.item():.4f}")
print(f"Total gradient norm: {gradient_norm:.4f}")
print("-> zero_grad -> forward -> loss.backward -> step is GradientTape + apply_gradients made explicit.")

In [ ]:
#  Full-batch training — fixed seed, fixed split, actual project measurements
# Trains one classifier and records loss/gradient history; zero_grad toggles accumulation for the ablation below
def train_classifier(hidden_width=HIDDEN_WIDTH, zero_grad=True, epochs=160):
    torch.manual_seed(SEED)
    candidate = AntarcticFieldGuide(hidden_width=hidden_width).to(device)
    optimizer = torch.optim.Adam(candidate.parameters(), lr=0.02)
    history, grad_history = [], []
    for _ in range(epochs):

        # Skipping zero_grad lets gradients accumulate across epochs, on purpose, for the ablation
        if zero_grad:
            optimizer.zero_grad()
        logits = candidate(X_train)
        loss = loss_fn(logits, y_train)
        loss.backward()
        total_grad = torch.sqrt(sum(p.grad.pow(2).sum() for p in candidate.parameters())).item()
        optimizer.step()
        history.append(loss.item())
        grad_history.append(total_grad)
    return candidate, np.array(history), np.array(grad_history)


model, loss_history, grad_history = train_classifier()

# Plot the training loss and gradient-norm curves side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(loss_history, color="#4C78A8")
axes[0].set(title="Antarctic Field Guide training loss", xlabel="Epoch", ylabel="Cross-entropy")
axes[1].plot(grad_history, color="#F58518")
axes[1].set(title="Gradient norm per epoch", xlabel="Epoch", ylabel="L2 norm")
plt.tight_layout()
plt.show()
print(f"Loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}")
print("-> The curve comes from this project's real data and this explicit optimizer loop.")

### Predict first — is `optimizer.zero_grad()` cosmetic?

Train two identically initialized models. One clears gradients on every step; the other does not.

1. Their learning curves will match because `backward()` overwrites gradients.
2. The no-clear model will accumulate prior gradients and follow a different update path.
3. The no-clear model will raise an error immediately.


In [ ]:
#  Controlled ablation — gradient accumulation changes the optimization trajectory
# Train two identical models, one clearing gradients each epoch, one letting them accumulate
clean_model, clean_loss, clean_grad = train_classifier(zero_grad=True, epochs=45)
accum_model, accum_loss, accum_grad = train_classifier(zero_grad=False, epochs=45)

# Plot loss and gradient-norm curves for both conditions side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(clean_loss, label="zero_grad each epoch", color="#4C78A8")
axes[0].plot(accum_loss, label="gradients accumulate", color="#E45756")
axes[0].set(title="Loss changes when gradients accumulate", xlabel="Epoch", ylabel="Loss")
axes[0].legend()
axes[1].plot(clean_grad, label="cleared", color="#4C78A8")
axes[1].plot(accum_grad, label="accumulated", color="#E45756")
axes[1].set(title="Gradient norm reveals the mechanism", xlabel="Epoch", ylabel="L2 norm")
axes[1].legend()
plt.tight_layout()
plt.show()
print(f"Final loss with zero_grad: {clean_loss[-1]:.3f}")
print(f"Final loss without it:     {accum_loss[-1]:.3f}")
print("-> PyTorch accumulates gradients by design; clearing them is a semantic part of each training step.")

#### What just happened — and what's missing

The project trained, and the ablation makes a framework behavior concrete: gradients accumulate unless you explicitly clear them. Training code is not inference code, though. The next part switches the model into evaluation mode and turns raw logits into a decision audit.


---

## Part 5 — Evaluation Is a Different Mode

### Keras reference — inference

```python
logits = model(X_test, training=False)
probabilities = keras.ops.softmax(logits, axis=-1)
predictions = keras.ops.argmax(probabilities, axis=-1)
```

In PyTorch, `model.eval()` controls evaluation behavior for layers such as dropout and batch normalization; `torch.no_grad()` prevents construction of an autograd graph for inference.


In [ ]:
#  Held-out evaluation — no gradient graph, probabilities only after the model output
model.eval()

# Disable autograd tracking for inference-only forward passes
with torch.no_grad():
    test_logits = model(X_test)
    test_probabilities = torch.softmax(test_logits, dim=1)
    test_predictions = test_logits.argmax(dim=1)
test_accuracy = (test_predictions == y_test).float().mean().item()
confusion = torch.zeros(len(species_names), len(species_names), dtype=torch.int64)

# Tally actual-vs-predicted species pairs into a confusion matrix
for actual, predicted in zip(y_test.cpu(), test_predictions.cpu()):
    confusion[actual, predicted] += 1

# Render the confusion matrix as an annotated heatmap
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(confusion.numpy(), annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=species_names, yticklabels=species_names, ax=ax)
ax.set(title="Held-out penguin species predictions", xlabel="Predicted species", ylabel="Actual species")
plt.tight_layout()
plt.show()
print(f"Held-out accuracy: {test_accuracy:.1%}")
print("-> eval() + no_grad() is PyTorch's explicit inference contract.")

In [ ]:
#  Prediction audit — inspect actual species decisions and confidence
print("Row | actual species | predicted species | confidence")

# Walk the first 8 held-out rows and print the model's decision and confidence for each
for local_index, source_index in enumerate(test_indices[:8]):
    actual = species_names[y_test[local_index].item()]
    predicted_id = test_predictions[local_index].item()
    predicted = species_names[predicted_id]
    confidence = test_probabilities[local_index, predicted_id].item()
    print(f"{source_index:3d} | {actual:13s} | {predicted:17s} | {confidence:.2f}")
print("-> A classifier supplies evidence, not a biological verdict; inspect failures before trusting it.")

---

## Part 6 — Controlled Project Decisions

### Your turn — hidden width

Change `WIDTHS` below, predict whether the widest network will improve held-out accuracy, then run the controlled comparison. The split, seed, epochs, optimizer, and feature set stay fixed; width is the only changed variable.


In [ ]:
#  CHANGE this list, then compare capacity with the same project conditions
WIDTHS = [4, 16, 64]
width_results = []

# Train and evaluate one classifier per hidden width, holding every other setting fixed
for width in WIDTHS:
    candidate, candidate_loss, _ = train_classifier(hidden_width=width, epochs=160)
    candidate.eval()
    with torch.no_grad():
        accuracy = (candidate(X_test).argmax(dim=1) == y_test).float().mean().item()
    width_results.append((width, candidate_loss[-1], accuracy))

# Bar-chart final loss and held-out accuracy side by side across widths
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar([str(width) for width, _, _ in width_results], [loss for _, loss, _ in width_results], color="#72B7B2")
axes[0].set(title="Final training loss", xlabel="Hidden width", ylabel="Loss")
axes[1].bar([str(width) for width, _, _, in width_results], [accuracy for _, _, accuracy in width_results], color="#54A24B")
axes[1].set(title="Held-out accuracy", xlabel="Hidden width", ylabel="Accuracy", ylim=(0, 1))
plt.tight_layout()
plt.show()
for width, loss, accuracy in width_results:
    print(f"width={width:2d}: loss={loss:.3f}, held-out accuracy={accuracy:.1%}")
print("-> More capacity can lower training loss without guaranteeing a better held-out decision rule.")

In [ ]:
#  Feature-set ablation — bill dimensions alone vs. all four measurements
# Trains a fresh classifier restricted to the given feature columns and returns held-out accuracy
def train_with_features(feature_columns):
    local_train, local_test = X_train[:, feature_columns], X_test[:, feature_columns]
    torch.manual_seed(SEED)
    candidate = nn.Sequential(nn.Linear(len(feature_columns), 16), nn.ReLU(), nn.Linear(16, 3)).to(device)
    optimizer = torch.optim.Adam(candidate.parameters(), lr=0.02)
    for _ in range(160):
        optimizer.zero_grad()
        loss = loss_fn(candidate(local_train), y_train)
        loss.backward()
        optimizer.step()
    candidate.eval()

    # Evaluate without building an autograd graph
    with torch.no_grad():
        return (candidate(local_test).argmax(dim=1) == y_test).float().mean().item()


# Compare a 2-feature model against the full 4-feature model, same training conditions
bill_accuracy = train_with_features([0, 1])
full_feature_accuracy = train_with_features(list(range(4)))

# Bar-chart the two conditions' held-out accuracy
plt.figure(figsize=(6, 4))
plt.bar(["Bill dimensions", "All measurements"], [bill_accuracy, full_feature_accuracy], color=["#B279A2", "#59A14F"])
plt.ylim(0, 1)
plt.ylabel("Held-out accuracy")
plt.title("Feature-set ablation: one changed variable")
plt.show()
print(f"Bill dimensions only:  {bill_accuracy:.1%}")
print(f"All four measurements: {full_feature_accuracy:.1%}")
print("-> The measured difference, not the story we hoped for, decides whether extra features earned their keep.")

---

## Summary — Keras Knowledge, PyTorch Vocabulary

| Familiar Keras operation | PyTorch operation you practiced | Why it matters |
|---|---|---|
| `set_random_seed` | seed Python, NumPy, PyTorch | Reproducible experiments require explicit ownership |
| `Dense` | `nn.Linear` | Same affine map; weight storage is transposed |
| `Sequential` | `nn.Sequential` inside `nn.Module` | Layers register and `forward()` defines computation |
| `from_logits=True` | `nn.CrossEntropyLoss` | Train on raw scores, not prematurely-softmaxed probabilities |
| `GradientTape` + `apply_gradients` | `zero_grad -> backward -> step` | Training becomes visible and debuggable |
| `predict(training=False)` | `eval()` + `no_grad()` | Inference has a distinct graph and behavior contract |

### Key insights to keep

- **PyTorch is not different theory:** it makes the contracts Keras can hide visible: dtype, device, graph lifetime, and gradient state.
- **`nn.Linear` and `Dense` compute the same affine function:** only their stored weight orientation differs.
- **Raw logits are an agreement with cross-entropy:** applying softmax twice is a quiet but meaningful bug.
- **`zero_grad()` is part of the algorithm:** PyTorch intentionally accumulates gradients until you clear them.
- **A useful project needs visible provenance:** Antarctic Field Guide uses a bundled CC0 dataset and removes incomplete records before training.

**Next:** continue to [`../04-cnns/convolutional-neural-networks.ipynb`](../04-cnns/convolutional-neural-networks.ipynb) for the optional vision branch, or [`../05-rnn-sequence-modeling/rnn-sequence-modeling.ipynb`](../05-rnn-sequence-modeling/rnn-sequence-modeling.ipynb) for the language-model route. Complete tokenization in prerequisite 06 before the PyTorch RNN bridge in prerequisite 07.